In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import numpy as np
from tqdm import tqdm

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Mounted at /content/drive
Using device: cuda


In [ ]:
!pip install transformers -q

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),(0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),(0.2023, 0.1994, 0.2010)),
])

full_train = torchvision.datasets.CIFAR10(root='./data', train=True,
                                           download=True, transform=transform_train)
test_set   = torchvision.datasets.CIFAR10(root='./data', train=False,
                                           download=True, transform=transform_test)

train_indices = np.load('/content/drive/MyDrive/ViT-Robustness/train_indices.npy')
val_indices   = np.load('/content/drive/MyDrive/ViT-Robustness/val_indices.npy')

train_set = torch.utils.data.Subset(full_train, train_indices)
val_set   = torch.utils.data.Subset(full_train, val_indices)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=64, shuffle=True)
val_loader   = torch.utils.data.DataLoader(val_set,   batch_size=64, shuffle=False)
test_loader  = torch.utils.data.DataLoader(test_set,  batch_size=64, shuffle=False)

100%|██████████| 170M/170M [00:06<00:00, 27.4MB/s]


In [ ]:
def train_model(model, train_loader, val_loader, epochs=10, lr=1e-4, save_path=None):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0
    history = {'train_loss': [], 'val_acc': []}

    for epoch in range(epochs):
        model.train()
        running_loss = 0
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        # Validation
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        val_acc = correct / total
        avg_loss = running_loss / len(train_loader)
        history['train_loss'].append(avg_loss)
        history['val_acc'].append(val_acc)
        print(f"Epoch {epoch+1}: Loss={avg_loss:.4f}, Val Acc={val_acc:.4f}")

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            if save_path:
                torch.save(model.state_dict(), save_path)
                print(f"  --> Saved best model (val acc: {val_acc:.4f})")

    return history

In [ ]:
import torchvision.models as models

# Load pretrained ResNet-18 and replace final layer for 10 classes
cnn_model = models.resnet18(pretrained=True)
cnn_model.fc = nn.Linear(cnn_model.fc.in_features, 10)
cnn_model = cnn_model.to(device)

cnn_save_path = '/content/drive/MyDrive/ViT-Robustness/checkpoints/resnet18_best.pth'
cnn_history = train_model(cnn_model, train_loader, val_loader,
                           epochs=10, lr=1e-4, save_path=cnn_save_path)
print("CNN training complete!")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 96.5MB/s]
Epoch 1/10: 100%|██████████| 704/704 [00:54<00:00, 12.82it/s]


Epoch 1: Loss=1.1653, Val Acc=0.7092
  --> Saved best model (val acc: 0.7092)


Epoch 2/10: 100%|██████████| 704/704 [00:38<00:00, 18.49it/s]


...

Epoch 8: Loss=0.4552, Val Acc=0.8214
  --> Saved best model (val acc: 0.8214)


Epoch 9/10: 100%|██████████| 704/704 [00:37<00:00, 18.61it/s]


Epoch 9: Loss=0.4254, Val Acc=0.8224
  --> Saved best model (val acc: 0.8224)


Epoch 10/10: 100%|██████████| 704/704 [00:37<00:00, 18.63it/s]


Epoch 10: Loss=0.4052, Val Acc=0.8278
  --> Saved best model (val acc: 0.8278)
CNN training complete!


In [ ]:
from transformers import ViTForImageClassification
import torchvision.transforms as transforms

# ViT needs 224x224 images, different transform
transform_vit = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5]),
])

full_train_vit = torchvision.datasets.CIFAR10(root='./data', train=True,
                                               download=True, transform=transform_vit)
test_set_vit   = torchvision.datasets.CIFAR10(root='./data', train=False,
                                               download=True, transform=transform_vit)

train_set_vit = torch.utils.data.Subset(full_train_vit, train_indices)
val_set_vit   = torch.utils.data.Subset(full_train_vit, val_indices)

train_loader_vit = torch.utils.data.DataLoader(train_set_vit, batch_size=32, shuffle=True)
val_loader_vit   = torch.utils.data.DataLoader(val_set_vit,   batch_size=32, shuffle=False)
test_loader_vit  = torch.utils.data.DataLoader(test_set_vit,  batch_size=32, shuffle=False)

# Load pretrained ViT
vit_model = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224',
    num_labels=10,
    ignore_mismatched_sizes=True
)
vit_model = vit_model.to(device)

# Slightly modified train function for HuggingFace model output format
def train_vit(model, train_loader, val_loader, epochs=10, lr=2e-5, save_path=None):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    best_val_acc = 0

    for epoch in range(epochs):
        model.train()
        running_loss = 0
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(pixel_values=images).logits
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(pixel_values=images).logits
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        val_acc = correct / total
        print(f"Epoch {epoch+1}: Loss={running_loss/len(train_loader):.4f}, Val Acc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            if save_path:
                torch.save(model.state_dict(), save_path)
                print(f"  --> Saved best ViT model")

vit_save_path = '/content/drive/MyDrive/ViT-Robustness/checkpoints/vit_best.pth'
train_vit(vit_model, train_loader_vit, val_loader_vit,
          epochs=10, lr=2e-5, save_path=vit_save_path)
print("ViT training complete!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                         
------------------+----------+-----------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([10])          
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([10, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.
Epoch 1/10: 100%|██████████| 1407/1407 [29:14<00:00,  1.25s/it]


...

Epoch 8/10: 100%|██████████| 1407/1407 [29:13<00:00,  1.25s/it]


Epoch 8: Loss=0.0086, Val Acc=0.9796


Epoch 9/10: 100%|██████████| 1407/1407 [29:14<00:00,  1.25s/it]


Epoch 9: Loss=0.0102, Val Acc=0.9832


Epoch 10/10:  20%|██        | 285/1407 [05:55<23:20,  1.25s/it]